In [1]:
# import pandas library, which will be used  for etl
import pandas as pd

In [2]:
# load CSV file into pandas DataFrames.
path = "Project2_dataset.csv"
Project_2 = pd.read_csv(path)

In [3]:
# use '.head()' to review tables
Project_2.head(5)

,Airline Name,Airline Country,Departure Airport Name,Departure Airport City,Departure Airport Country/Region,Arrival Airport Name,Arrival Airport City,Arrival Airport Country/Region,Plane Name
0,Tuna Aero,Sweden,Norsup Airport,Norsup,Vanuatu,Bauerfield International Airport,Port-vila,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
1,Tuna Aero,Sweden,Santo Pekoa International Airport,Santo,Vanuatu,Lonorore Airport,Lonorore,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
2,Tuna Aero,Sweden,Craig Cove Airport,Craig Cove,Vanuatu,Norsup Airport,Norsup,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
3,Tuna Aero,Sweden,Bauerfield International Airport,Port-vila,Vanuatu,Norsup Airport,Norsup,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
4,Tuna Aero,Sweden,Aneityum Airport,Anelghowhat,Vanuatu,Tanna Airport,Tanna,Vanuatu,Harbin Yunshuji Y12;Pilatus Britten-Norman BN-...


In [4]:
# check the shape of the dataset
# has over 57,300 rows and 9 variables
Project_2.shape

(57301, 9)

In [5]:
# check data types
Project_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57301 entries, 0 to 57300
Data columns (total 9 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Airline Name                      57301 non-null  object
 1   Airline Country                   57301 non-null  object
 2   Departure Airport Name            57301 non-null  object
 3   Departure Airport City            57301 non-null  object
 4   Departure Airport Country/Region  57301 non-null  object
 5   Arrival Airport Name              57301 non-null  object
 6   Arrival Airport City              57301 non-null  object
 7   Arrival Airport Country/Region    57301 non-null  object
 8   Plane Name                        57301 non-null  object
dtypes: object(9)
memory usage: 3.9+ MB


In [6]:
# check for nulls
# no nulls for now, will check again after cleaning
Project_2.isnull().sum()

Airline Name                        0
Airline Country                     0
Departure Airport Name              0
Departure Airport City              0
Departure Airport Country/Region    0
Arrival Airport Name                0
Arrival Airport City                0
Arrival Airport Country/Region      0
Plane Name                          0
dtype: int64

In [7]:
# apply standard etl cleaning(removing duplicates and trimming white spaces)
Project_2.drop_duplicates()
Project_2_cleaned = Project_2.copy()
projec_2_col = Project_2.columns.tolist()
for col in projec_2_col:
    Project_2_cleaned[col] = Project_2[col].astype(str).str.strip()
Project_2_cleaned.head(5)

,Airline Name,Airline Country,Departure Airport Name,Departure Airport City,Departure Airport Country/Region,Arrival Airport Name,Arrival Airport City,Arrival Airport Country/Region,Plane Name
0,Tuna Aero,Sweden,Norsup Airport,Norsup,Vanuatu,Bauerfield International Airport,Port-vila,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
1,Tuna Aero,Sweden,Santo Pekoa International Airport,Santo,Vanuatu,Lonorore Airport,Lonorore,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
2,Tuna Aero,Sweden,Craig Cove Airport,Craig Cove,Vanuatu,Norsup Airport,Norsup,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
3,Tuna Aero,Sweden,Bauerfield International Airport,Port-vila,Vanuatu,Norsup Airport,Norsup,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
4,Tuna Aero,Sweden,Aneityum Airport,Anelghowhat,Vanuatu,Tanna Airport,Tanna,Vanuatu,Harbin Yunshuji Y12;Pilatus Britten-Norman BN-...


In [8]:
# add a surrogate key to identify each unique row
Project_2_cleaned['Flight_ID'] = ['FL'+str(i+1) for i in range(len(Project_2_cleaned))]
# expand the plane name column by removing the ';', then populating each plane name into it's own row and removed any empty spaces
Project_2_cleaned['Plane Name'] = Project_2_cleaned['Plane Name'].str.split(';')
Project_2_cleaned = Project_2_cleaned.explode('Plane Name').reset_index(drop=True)
Project_2_cleaned['Plane Name'] = Project_2_cleaned['Plane Name'].str.strip()
# check if correctly plane names have been correctly expanded
Project_2_cleaned.head(5)

,Airline Name,Airline Country,Departure Airport Name,Departure Airport City,Departure Airport Country/Region,Arrival Airport Name,Arrival Airport City,Arrival Airport Country/Region,Plane Name,Flight_ID
0,Tuna Aero,Sweden,Norsup Airport,Norsup,Vanuatu,Bauerfield International Airport,Port-vila,Vanuatu,Harbin Yunshuji Y12,FL1
1,Tuna Aero,Sweden,Norsup Airport,Norsup,Vanuatu,Bauerfield International Airport,Port-vila,Vanuatu,De Havilland Canada DHC-6 Twin Otter,FL1
2,Tuna Aero,Sweden,Norsup Airport,Norsup,Vanuatu,Bauerfield International Airport,Port-vila,Vanuatu,Pilatus Britten-Norman BN-2A/B Islander,FL1
3,Tuna Aero,Sweden,Santo Pekoa International Airport,Santo,Vanuatu,Lonorore Airport,Lonorore,Vanuatu,Harbin Yunshuji Y12,FL2
4,Tuna Aero,Sweden,Santo Pekoa International Airport,Santo,Vanuatu,Lonorore Airport,Lonorore,Vanuatu,De Havilland Canada DHC-6 Twin Otter,FL2


In [9]:
# remove duplicates and keep unique records of airlines
airline_df = Project_2_cleaned.drop_duplicates(subset=['Airline Name', 'Airline Country']).reset_index(drop=True)
# add surrogate key
airline_df['Airline_ID'] = ['AL'+str(i+1) for i in range(len(airline_df))]
# select preserved fields
airline_df = airline_df[[
    'Airline_ID',
    'Airline Name',
    'Airline Country'
]]
airline_df.head(5)

,Airline_ID,Airline Name,Airline Country
0,AL1,Tuna Aero,Sweden
1,AL2,Cranwell Flying Training Unit,United Kingdom
2,AL3,Renan,Moldova
3,AL4,Voar Lda,Angola
4,AL5,Southern Cargo Air Lines,Russia


In [10]:
# combining both ariport columns and drop duplicate keys to only keep unique airlines
dep_airports = Project_2_cleaned[['Departure Airport Name', 'Departure Airport City', 'Departure Airport Country/Region']]
dep_airports.columns = ['Airport Name', 'Airport City', 'Airport Country/Region']
arr_airports = Project_2_cleaned[['Arrival Airport Name', 'Arrival Airport City', 'Arrival Airport Country/Region']]
arr_airports.columns = ['Airport Name', 'Airport City', 'Airport Country/Region']
airport_df = pd.concat([arr_airports, dep_airports]).drop_duplicates().reset_index(drop=True)
# add surrogate key
airport_df['Airport_ID'] = ['AP'+str(i+1) for i in range(len(airport_df))]
# select preserved fields
airport_df = airport_df[[
    'Airport_ID',
    'Airport Name',
    'Airport City',
    'Airport Country/Region'
]]
# check if airport correctly populated
airport_df.head(5)

,Airport_ID,Airport Name,Airport City,Airport Country/Region
0,AP1,Bauerfield International Airport,Port-vila,Vanuatu
1,AP2,Lonorore Airport,Lonorore,Vanuatu
2,AP3,Norsup Airport,Norsup,Vanuatu
3,AP4,Tanna Airport,Tanna,Vanuatu
4,AP5,Santo Pekoa International Airport,Santo,Vanuatu


In [11]:
# combining both airplane columns
airplane_df = Project_2_cleaned[['Plane Name']]
#remove duplicates
airplane_df = airplane_df.drop_duplicates().reset_index(drop=True)
# add surrogate key
airplane_df['Airplane_ID'] = ['AR'+str(i+1) for i in range(len(airplane_df))]
# select preserved fields
airplane_df = airplane_df[[
    'Airplane_ID',
    'Plane Name'
]]
# check if airplane correctly populated
airplane_df.head(5)

,Airplane_ID,Plane Name
0,AR1,Harbin Yunshuji Y12
1,AR2,De Havilland Canada DHC-6 Twin Otter
2,AR3,Pilatus Britten-Norman BN-2A/B Islander
3,AR4,Yakovlev Yak-40
4,AR5,Yakovlev Yak-42


In [12]:
# populate the flight node
flight_df = Project_2_cleaned[['Flight_ID']].drop_duplicates().reset_index(drop=True)
flight_df.head(5)

,Flight_ID
0,FL1
1,FL2
2,FL3
3,FL4
4,FL5


In [13]:
# relationship: flight OPERATED_BY airline
# merge by extracting the flight id and airline id
rel_operated_by = Project_2_cleaned.merge(airline_df, on='Airline Name')[['Flight_ID', 'Airline_ID']].drop_duplicates().reset_index(drop=True)
rel_operated_by.head(5)

,Flight_ID,Airline_ID
0,FL1,AL1
1,FL2,AL1
2,FL3,AL1
3,FL4,AL1
4,FL5,AL1


In [14]:
# relationship: flight DEPARTS_FROM airline
# merge by extracting the flight id and airport id
rel_departs_from = Project_2_cleaned.merge(airport_df,
                                           left_on=['Departure Airport Name', 'Departure Airport City', 'Departure Airport Country/Region'],
                                           right_on=['Airport Name', 'Airport City', 'Airport Country/Region'])[['Flight_ID', 'Airport_ID']].drop_duplicates().reset_index(drop=True)
rel_departs_from.head(5)

,Flight_ID,Airport_ID
0,FL1,AP3
1,FL6191,AP3
2,FL6192,AP3
3,FL6346,AP3
4,FL6347,AP3


In [15]:
# relationship: flight ARRIVES_AT airport
# merge by extracting the flight id and airport id
rel_arrives_at = Project_2_cleaned.merge(airport_df,
                                           left_on=['Arrival Airport Name', 'Arrival Airport City', 'Arrival Airport Country/Region'],
                                           right_on=['Airport Name', 'Airport City', 'Airport Country/Region'])[['Flight_ID', 'Airport_ID']].drop_duplicates().reset_index(drop=True)
rel_arrives_at.head(5)

,Flight_ID,Airport_ID
0,FL1,AP1
1,FL12,AP1
2,FL16,AP1
3,FL20,AP1
4,FL6337,AP1


In [16]:
# relationship: flight USES an airplane
# merge by extracting flight id and plane id
rel_uses = Project_2_cleaned.merge(airplane_df, on = 'Plane Name')[['Flight_ID', 'Airplane_ID']].drop_duplicates().reset_index(drop=True)
rel_uses.head(5)

,Flight_ID,Airplane_ID
0,FL1,AR1
1,FL2,AR1
2,FL3,AR1
3,FL4,AR1
4,FL5,AR1


In [17]:
# relationship: one airport to another airport is a route
# merge by extracting airport id from one airport to another airport
rel_routes = rel_departs_from.merge(rel_arrives_at, on = 'Flight_ID')
rel_routes = rel_routes[['Airport_ID_x', 'Airport_ID_y']].rename(columns={'Airport_ID_x': 'A1_ID', 'Airport_ID_y': 'A2_ID'})
rel_routes.head(5)

,A1_ID,A2_ID
0,AP3,AP1
1,AP3,AP13
2,AP3,AP5
3,AP3,AP1255
4,AP3,AP1256


In [18]:
# export all the nodes and relationships
airline_df.to_csv('airline_node.csv', index = False)
airport_df.to_csv('airport_node.csv', index = False)
airplane_df.to_csv('airplane_node.csv', index = False)
flight_df.to_csv('flight_node.csv', index = False)
rel_operated_by.to_csv('rel_operated_by.csv', index = False)
rel_departs_from.to_csv('rel_departs_from.csv', index = False)
rel_arrives_at.to_csv('rel_arrives_at.csv', index = False)
rel_uses.to_csv('rel_uses.csv', index = False)
rel_routes.to_csv('rel_routes.csv', index = False)